# A4 — Évaluer, généraliser, communiquer

| Projet | Étape | Durée | Modules réinvestis |
|---|---|---|---|
| A — Recommandation de films | 4 sur 4 | 4 h | M1, M5, M6, M8 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Les étapes A1 à A3 ont produit un modèle et des réglages, choisis sur le paquet de validation. Il reste à livrer à la médiathèque :

1. une **évaluation honnête** : le paquet de test, sous scellés depuis A1, est enfin ouvert, une seule fois ;
2. une réponse à la question de départ, celle des **spectateurs qu'on connaît peu** : un nouvel abonné note quelques films à son inscription ; que valent les recommandations qu'on peut lui faire ?
3. un **module Python** fiable et testé, que la développeuse du service pourra utiliser ;
4. un **rapport** et une **présentation**.

**Activité** : Pourquoi ne faut-il ouvrir le paquet de test qu'une seule fois ? Que perdrait-on à revenir modifier les réglages après l'avoir regardé ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_reqm_test = 0.80                 # erreur du modèle à facteurs sur le paquet de test (modifiez)
pari_notes_pour_nouvel_abonne = 5     # nombre de notes à demander à l'inscription pour de bonnes prévisions (modifiez)

## ③ Abstraire

**Deux sortes d'erreurs.** L'erreur mesurée sur le test dépend du hasard du découpage : c'est une erreur **aléatoire**, qu'on estime en recommençant avec d'autres découpages. D'autres erreurs vont toujours dans le même sens, quel que soit le découpage : ce sont des erreurs **systématiques**. Par exemple, les spectateurs ne notent pas les films au hasard, mais surtout ceux qu'ils ont choisi de voir.

**Prévoir une note ou bien classer ?** Pour recommander, l'important n'est pas la note exacte, mais l'**ordre** : les films conseillés en premier doivent plaire. On mesure donc aussi, pour chaque spectateur, la proportion de films vraiment appréciés (note d'au moins 4) parmi les 5 films de son paquet de test que le modèle classe en tête.

**Un nouvel abonné.** Les profils des films $\mathbf{q}_f$ sont connus. Si un nouvel abonné note quelques films, on obtient son biais $b_u$ et ses goûts $\mathbf{p}_u$ par **un seul** petit problème de moindres carrés régularisés (module M6), sans tout réajuster :

$$
\left(A^{\mathsf{T}}A + \Lambda\right)\begin{pmatrix} b_u \\ \mathbf{p}_u \end{pmatrix} = A^{\mathsf{T}}\left(\mathbf{r} - \mu - \mathbf{b}_f\right),
\qquad
A = \begin{pmatrix} 1 & \mathbf{q}_{f_1}^{\mathsf{T}} \\ \vdots & \vdots \\ 1 & \mathbf{q}_{f_n}^{\mathsf{T}} \end{pmatrix},
$$

où $\Lambda$ est diagonale et contient les coefficients de régularisation.

**Activité** : Avec une seule note, combien d'inconnues faut-il trouver pour un modèle à 10 facteurs ? Que se passerait-il sans régularisation ? Que fait la régularisation dans ce cas ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Empaqueter le modèle

Toute la méthode des étapes A1 et A3 tient dans une **classe** : on l'ajuste une fois, puis on lui demande des prévisions, des recommandations, ou d'ajouter un spectateur. Chaque méthode vérifie ses entrées et refuse clairement ce qu'elle ne sait pas traiter.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from films import charger_films, decouper, reqm, Index, ajuster_biais, predire_biais

class Recommandeur:
    """Recommandation de films par biais et facteurs latents (moindres carrés alternés).

    Utilisation :
        modele = Recommandeur(k=10, lam=5).ajuster(notes)     # notes : userId, movieId, rating
        modele.predire([1, 1], [10, 20])                     # notes prévues
        modele.recommander(1, n=10)                          # identifiants des films conseillés
        nouveau = modele.ajouter_spectateur({10: 4.5, 20: 2.0})
    """

    COLONNES = ("userId", "movieId", "rating")

    def __init__(self, k=10, lam=5.0, lam_biais=10.0, iterations=15, graine=0):
        if k < 1 or lam < 0 or lam_biais < 0 or iterations < 1:
            raise ValueError("Réglages invalides : il faut k ≥ 1, lam ≥ 0, lam_biais ≥ 0, iterations ≥ 1.")
        self.k, self.lam, self.lam_biais, self.iterations, self.graine = k, lam, lam_biais, iterations, graine
        self.ajuste = False

    # ---------------------------------------------------------------- apprentissage
    def ajuster(self, notes):
        manquantes = [c for c in self.COLONNES if c not in notes.columns]
        if manquantes:
            raise ValueError(f"Colonnes manquantes : {manquantes}")
        if len(notes) == 0:
            raise ValueError("Aucune note fournie.")
        if notes["rating"].isna().any() or (notes["rating"] < 0.5).any() or (notes["rating"] > 5).any():
            raise ValueError("Les notes doivent être comprises entre 0,5 et 5.")
        self.mu, bu, bf = ajuster_biais(notes, self.lam_biais)
        self.index = Index(notes)
        self.bu = np.array([bu[u] for u in self.index.spectateurs])
        self.bf = np.array([bf[f] for f in self.index.films])
        R = self.index.matrice(notes)
        connu = ~np.isnan(R)
        E = np.where(connu, R - self.mu - self.bu[:, None] - self.bf[None, :], 0.0)
        self.P, self.Q = self._alterner(E, connu)
        self.vus = {u: set(g) for u, g in notes.groupby("userId")["movieId"]}
        self.nouveaux = {}
        self.ajuste = True
        return self

    def _alterner(self, E, connu):
        rng = np.random.default_rng(self.graine)
        P = rng.normal(0, 0.1, (E.shape[0], self.k))
        Q = rng.normal(0, 0.1, (E.shape[1], self.k))
        I = self.lam * np.eye(self.k)
        for _ in range(self.iterations):
            for u in range(E.shape[0]):
                j = connu[u]
                P[u] = np.linalg.solve(Q[j].T @ Q[j] + I, Q[j].T @ E[u, j])
            for f in range(E.shape[1]):
                i = connu[:, f]
                Q[f] = np.linalg.solve(P[i].T @ P[i] + I, P[i].T @ E[i, f])
        return P, Q

    # ---------------------------------------------------------------- prévision
    def _verifier(self):
        if not self.ajuste:
            raise RuntimeError("Le modèle n'est pas ajusté : appelez d'abord ajuster(notes).")

    def _profil(self, u):
        # renvoie (biais, vecteur de goûts) d'un spectateur, connu, ajouté ou inconnu
        if u in self.nouveaux:
            return self.nouveaux[u]
        i = self.index.ligne.get(u)
        if i is None:
            return 0.0, np.zeros(self.k)           # spectateur inconnu : on ne sait rien de ses goûts
        return self.bu[i], self.P[i]

    def predire(self, user_ids, movie_ids):
        self._verifier()
        user_ids, movie_ids = np.atleast_1d(user_ids), np.atleast_1d(movie_ids)
        if len(user_ids) != len(movie_ids):
            raise ValueError("user_ids et movie_ids doivent avoir la même longueur.")
        resultat = np.empty(len(user_ids))
        for n, (u, f) in enumerate(zip(user_ids, movie_ids)):
            b_u, p_u = self._profil(u)
            j = self.index.colonne.get(f)
            if j is None:                               # film inconnu : moyenne corrigée du spectateur
                resultat[n] = self.mu + b_u
            else:
                resultat[n] = self.mu + b_u + self.bf[j] + p_u @ self.Q[j]
        return np.clip(resultat, 0.5, 5)

    def recommander(self, user_id, n=10, notes_minimum=0, nb_notes_films=None):
        """Les n films non vus à la meilleure note prévue (éventuellement parmi les films assez notés)."""
        self._verifier()
        if n < 1:
            raise ValueError("n doit être au moins 1.")
        b_u, p_u = self._profil(user_id)
        scores = self.mu + b_u + self.bf + self.Q @ p_u
        deja_vus = self.vus.get(user_id, set())
        candidats = [j for j, f in enumerate(self.index.films) if f not in deja_vus
                     and (nb_notes_films is None or nb_notes_films.get(f, 0) >= notes_minimum)]
        meilleurs = sorted(candidats, key=lambda j: -scores[j])[:n]
        return [self.index.films[j] for j in meilleurs]

    def ajouter_spectateur(self, notes_du_spectateur, identifiant=None):
        """Ajoute un spectateur à partir de quelques notes {movieId: note}, sans tout réajuster.

        Ses biais et ses goûts sont obtenus par moindres carrés régularisés, les profils des films étant fixés.
        """
        self._verifier()
        connus = {f: r for f, r in notes_du_spectateur.items() if f in self.index.colonne}
        if not connus:
            raise ValueError("Aucun des films notés n'est connu du modèle.")
        j = np.array([self.index.colonne[f] for f in connus])
        r = np.array(list(connus.values()), dtype=float)
        A = np.column_stack([np.ones(len(j)), self.Q[j]])                 # inconnues : b_u puis p_u
        regul = np.diag([self.lam_biais] + [self.lam] * self.k)
        x = np.linalg.solve(A.T @ A + regul, A.T @ (r - self.mu - self.bf[j]))
        if identifiant is None:
            identifiant = f"nouveau-{len(self.nouveaux) + 1}"
        self.nouveaux[identifiant] = (x[0], x[1:])
        self.vus[identifiant] = set(connus)
        return identifiant

In [ ]:
# Quelques tests rapides sur un exemple dont on connaît la réponse : deux « camps » de goûts opposés
mini = pd.DataFrame([(1, 10, 5), (1, 11, 4.5), (1, 12, 1), (2, 10, 4.5), (2, 11, 5), (2, 13, 1.5),
                     (3, 12, 5), (3, 13, 4.5), (3, 10, 1), (4, 12, 4.5), (4, 13, 5), (4, 11, 1.5)],
                    columns=["userId", "movieId", "rating"])
m = Recommandeur(k=2, lam=0.1, iterations=30).ajuster(mini)
assert m.predire([1], [13])[0] < m.predire([1], [11])[0], "le spectateur 1 devrait préférer le film 11"
assert set(m.recommander(1, n=5)) == {13}, "seul le film 13 n'a pas été vu par le spectateur 1"
nouveau = m.ajouter_spectateur({10: 5, 11: 4.5})
assert m.predire([nouveau], [11])[0] > m.predire([nouveau], [13])[0]
try:
    Recommandeur().predire([1], [1])
    raise AssertionError("un modèle non ajusté devrait refuser de prévoir")
except RuntimeError:
    pass
print("Tests réussis")

### L'évaluation finale

On ajuste le modèle sur **toutes** les notes hors test (apprentissage et validation réunis), avec les réglages choisis à l'étape A3, puis on ouvre le paquet de test.

In [ ]:
REGLAGES = dict(k=10, lam=5.0, lam_biais=10.0, iterations=15)     # reportez ici vos réglages de l'étape A3

notes, films, source = charger_films()
reste, test = decouper(notes, part_test=0.2, graine=0)
debut = time.perf_counter()
modele = Recommandeur(**REGLAGES).ajuster(reste)
print(f"Ajustement sur {len(reste)} notes en {time.perf_counter() - debut:.1f} s")

pred_facteurs = modele.predire(test["userId"].to_numpy(), test["movieId"].to_numpy())
pred_biais = np.clip(predire_biais(ajuster_biais(reste, 10), test), 0.5, 5)
pred_moyenne = np.full(len(test), reste["rating"].mean())
final = pd.Series({"moyenne générale": reqm(test["rating"], pred_moyenne),
                   "biais": reqm(test["rating"], pred_biais),
                   "biais + facteurs": reqm(test["rating"], pred_facteurs)})
print(final.round(4).to_string())

nb_notes = reste.groupby("userId").size()
classe = pd.cut(test["userId"].map(nb_notes), [0, 30, 60, 150, 10_000], labels=["≤ 30", "31 à 60", "61 à 150", "> 150"])
par_classe = pd.DataFrame({"biais": (test["rating"] - pred_biais) ** 2, "facteurs": (test["rating"] - pred_facteurs) ** 2})
print("\nREQM selon l'activité du spectateur :")
print(np.sqrt(par_classe.groupby(classe, observed=True).mean()).round(4).to_string())

### L'erreur aléatoire : changer de découpage

In [ ]:
erreurs = []
for graine in (10, 11, 12):
    r_, t_ = decouper(notes, part_test=0.2, graine=graine)
    m_ = Recommandeur(**REGLAGES).ajuster(r_)
    erreurs.append(reqm(t_["rating"], m_.predire(t_["userId"].to_numpy(), t_["movieId"].to_numpy())))
print("REQM sur trois autres découpages :", np.round(erreurs, 4))
print(f"Étendue : {max(erreurs) - min(erreurs):.4f} étoile")

### Classer : les films placés en tête plaisent-ils ?

In [ ]:
nb_notes_film = reste.groupby("movieId").size()
evaluation = test.assign(facteurs=pred_facteurs, biais=pred_biais,
                         popularite=test["movieId"].map(nb_notes_film).fillna(0).to_numpy(),
                         hasard=np.random.default_rng(0).random(len(test)))
evaluation = evaluation[evaluation.groupby("userId")["rating"].transform("size") >= 10]

def precision_en_tete(colonne, n=5):
    tetes = evaluation.sort_values(colonne, ascending=False).groupby("userId").head(n)
    return (tetes["rating"] >= 4).mean()

classement = pd.Series({c: precision_en_tete(c) for c in ["hasard", "popularite", "biais", "facteurs"]})
print(f"{evaluation['userId'].nunique()} spectateurs ayant au moins 10 films dans le test")
print("Part de films appréciés (note ≥ 4) parmi les 5 films classés en tête :")
print(classement.round(3).to_string())

**Activité** : Le classement par popularité fait-il beaucoup mieux que le hasard ? Pourquoi, d'après vous ? Est-ce une qualité de la popularité, ou une conséquence de la façon dont les données ont été produites ?

Indiquez vos réponses et justifiez-les.

### Les spectateurs qu'on connaît peu

On retire complètement 40 spectateurs actifs des données d'apprentissage. Puis on les « inscrit » comme de nouveaux abonnés avec seulement $n$ de leurs notes, et l'on prévoit toutes les autres.

In [ ]:
rng = np.random.default_rng(3)
actifs = nb_notes[nb_notes >= 80].index
cobayes = rng.choice(actifs, 40, replace=False)
sans_cobayes = reste[~reste["userId"].isin(cobayes)]
modele_froid = Recommandeur(**REGLAGES).ajuster(sans_cobayes)
moyenne_film = sans_cobayes.groupby("movieId")["rating"].mean()

tailles = [1, 3, 5, 10, 20, 40]
courbe = {"biais et facteurs": [], "moyenne du film": []}
for n in tailles:
    err_f, err_m = [], []
    for u in cobayes:
        siennes = reste[(reste["userId"] == u) & reste["movieId"].isin(modele_froid.index.colonne)]   # films connus du modèle
        siennes = siennes.sample(frac=1, random_state=int(u))
        connues, a_prevoir = siennes.iloc[:n], siennes.iloc[n:]
        ident = modele_froid.ajouter_spectateur(dict(zip(connues["movieId"], connues["rating"])), identifiant=f"c{u}-{n}")
        err_f.append((a_prevoir["rating"] - modele_froid.predire([ident] * len(a_prevoir), a_prevoir["movieId"].to_numpy())) ** 2)
        err_m.append((a_prevoir["rating"] - a_prevoir["movieId"].map(moyenne_film)) ** 2)
    courbe["biais et facteurs"].append(np.sqrt(pd.concat(err_f).mean()))
    courbe["moyenne du film"].append(np.sqrt(pd.concat(err_m).mean()))

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(tailles, courbe["biais et facteurs"], "o-", color=LIOUAI["vert"], label="biais et facteurs, nouvel abonné")
ax.plot(tailles, courbe["moyenne du film"], "s--", color=LIOUAI["gris"], label="moyenne du film (aucune personnalisation)")
ax.set_xscale("log"); ax.set_xticks(tailles, tailles)
ax.set_xlabel("nombre de notes données à l'inscription"); ax.set_ylabel("REQM sur ses autres films")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(pd.DataFrame(courbe, index=tailles).round(4).to_string())

### Essayer soi-même

Modifiez le dictionnaire ci-dessous : il contient vos notes pour quelques films parmi les plus notés. La cellule affiche ensuite vos recommandations.

In [ ]:
infos = films.set_index("movieId")
populaires = nb_notes_film.sort_values(ascending=False).index[:12]
print("Quelques films très notés :")
for f in populaires:
    print(f"  {f:6d}  {infos.loc[f, 'title']}  ({infos.loc[f, 'genres']})")

mes_notes = {int(populaires[0]): 5.0, int(populaires[1]): 4.5, int(populaires[2]): 1.5,
             int(populaires[3]): 2.0, int(populaires[4]): 4.0}                         # (modifiez)
moi = modele.ajouter_spectateur(mes_notes, identifiant="moi")
print("\nVos recommandations (films d'au moins 20 notes) :")
for f in modele.recommander(moi, n=8, notes_minimum=20, nb_notes_films=nb_notes_film.to_dict()):
    print(f"  {modele.predire([moi], [f])[0]:.2f}  {infos.loc[f, 'title']}  ({infos.loc[f, 'genres']})")

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : REQM test {pari_reqm_test} ; mesurée : {final['biais + facteurs']:.3f}")
seuil = courbe["biais et facteurs"][-1] + 0.5 * (courbe["biais et facteurs"][0] - courbe["biais et facteurs"][-1])
n_suffisant = next(n for n, v in zip(tailles, courbe["biais et facteurs"]) if v <= seuil)
print(f"Votre pari : {pari_notes_pour_nouvel_abonne} notes à l'inscription ; "
      f"la moitié du gain possible est atteinte dès {n_suffisant} notes")

**Activité** :

1. L'erreur de test est-elle proche de l'erreur de validation des étapes précédentes ? Plus grande, plus petite ? Comparez avec l'étendue due au découpage.
2. Combien de films conseilleriez-vous de demander à un nouvel abonné à son inscription ? Justifiez avec la courbe.
3. Citez une erreur systématique possible du modèle, que ni le test ni les changements de découpage ne peuvent révéler.
4. Formulez la réponse à la question de départ, en trois phrases, pour la responsable de la médiathèque.

Indiquez vos réponses et justifiez-les.

### Le rapport

Rédigez un rapport de **4 à 6 pages**, à partir de vos notebooks, en suivant ce plan :

1. **La question** : reformulation, découpage en sous-questions, hypothèses.
2. **Les données** : origine, licence, ce qu'elles contiennent et ce qu'elles ne disent pas.
3. **Les méthodes** : biais, voisins, facteurs ; une phrase et un schéma pour chacune, sans formule inutile.
4. **Les résultats** : un tableau des erreurs (validation et test), la courbe des nouveaux abonnés, un exemple de recommandations.
5. **La fiabilité** : erreurs aléatoires et systématiques, limites, ce qu'il faudrait vérifier avec de vraies données d'usage.
6. **La recommandation** à la médiathèque, et le mode d'emploi du module.

Indiquez vos réponses et justifiez-les.

### La présentation (10 minutes)

Préparez **6 diapositives au plus** : la question, les données en une image, l'idée des facteurs latents en une image, le tableau des résultats, la réponse pour les nouveaux abonnés, les limites. Une idée par diapositive ; les chiffres arrondis à ce qui est utile.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Ce que l'erreur ne mesure pas.** Un modèle peut avoir une très bonne erreur et ne recommander que des films que tout le monde connaît déjà. Comment mesurer la **diversité** ou la **nouveauté** des recommandations ?
- **Bulle de filtres.** Si les abonnés ne regardent plus que ce que le service leur recommande, que deviennent les données des années suivantes ?
- **Généraliser : le module.** Copiez la classe `Recommandeur` dans un fichier `recommandation.py`, à côté de `films.py`, et écrivez un fichier `test_recommandation.py` qui teste au moins : des prévisions toujours entre 0,5 et 5 ; un spectateur ou un film inconnu ; un nouvel abonné ; des entrées invalides. Le corrigé se trouve dans le dossier `corrige`.

In [ ]:
# À compléter : créez recommandation.py et test_recommandation.py, puis lancez vos tests ici

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
import importlib, sys
sys.path.insert(0, "corrige")
tests = importlib.import_module("test_recommandation")
for nom in sorted(dir(tests)):
    if nom.startswith("test_"):
        getattr(tests, nom)()
        print("ok", nom)

# Nouveauté : popularité moyenne des films recommandés, comparée à celle des films du catalogue
recos = [f for u in list(modele.index.spectateurs[:100]) for f in modele.recommander(u, n=10)]
print(f"\nNombre moyen de notes des films recommandés : {nb_notes_film.reindex(recos).fillna(0).mean():.0f} "
      f"(catalogue : {nb_notes_film.mean():.0f})")
print(f"Films différents recommandés aux 100 premiers spectateurs : {len(set(recos))} sur {len(recos)} recommandations")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Implémenter un modèle général sous forme de programme robuste": 1,
    "Corroborer ses résultats": 1,
    "Identifier erreurs systématiques et aléatoires": 1,
    "Structurer un rapport ou une présentation": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "A4-evaluer-et-livrer", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")